# Step 1 : Create Input

In [36]:
import torch

tokens = torch.tensor([
    [5,12,89],
    [7,25,41]
])

print(tokens.shape)

# Batch Size = 2

# Sequence Length = 3

# 2 sentences

# 3 tokens per sentence

torch.Size([2, 3])


# Step 2 : Embedding Layer

In [37]:
import torch.nn as nn
embedding=nn.Embedding(
  num_embeddings=10000,
  embedding_dim=128
)

# Token 0 -> row 0

# Token 1 -> row 1

# Token 2 -> row 2

# ...

# Token 9999 -> row 9999

# Step 3 : Lookup Embeddings

In [38]:
x=embedding(tokens)
print(x.shape)

#Meaning:

# Batch Size = 2

# Sequence Length = 3

# Embedding Dimension = 128

torch.Size([2, 3, 128])


# PART 2 : Q, K, V Generation

## Step 1

In [39]:
Wq = nn.Linear(128,128)

Wk = nn.Linear(128,128)

Wv = nn.Linear(128,128)

# What does nn.Linear(128,128) create?

# Weight matrix:

# (128,128)

# Bias:

# (128)

# Purpose:

# Input Embedding

# ↓

# Query Representation

# for Wq.

# Similarly for K and V.
Wq

Linear(in_features=128, out_features=128, bias=True)

# Step 2

Generate Q,K,V.

In [40]:
Q = Wq(x)

K = Wk(x)

V = Wv(x)

# Input shape:

# (2,3,128)

# Output shape:

# (2,3,128)

# Why no shape change?

# Because:

# 128 -> 128

# projection.

# Only values change.

# At this point:

# Input
#  ↓

# Q
# K
# V

# exist.

# No attention has happened yet.

# PART 3 : Single Head Attention

## Step 1

Transpose K.

In [41]:
K_t = K.transpose(-2,-1)

## Step 2

Compute Scores.

In [42]:
scores = Q @ K_t

# Every token compares itself

# with

# every token

## Step 3

Scale.

In [43]:
import math

scores = scores / math.sqrt(128)

# Reason:

# Large dot products make softmax unstable.

# Step 4

Apply Softmax.

In [44]:
import torch.nn.functional as F

attention_weights = F.softmax(
    scores,
    dim=-1
)

# Shape:

# (2,3,3)

# Meaning:

# Each row now contains probabilities.

# Rows sum to:

# 1

## Step 5

Multiply by V.

In [45]:
output = attention_weights @ V

# (2,3,3)

# @

# (2,3,128)

# =

# (2,3,128)

### Now we have completed an entire working:

Single Head Attention

from scratch.

# PART 4 : Multi Head Attention

# Step 1 : Create MultiHeadAttention Class

In [46]:
# What Happens Here?

# Suppose:

# embed_dim = 128
# num_heads = 8

# Then:

# head_dim = 128 // 8

# becomes:

# 16

# Meaning:

# Head 1 → 16 dimensions

# Head 2 → 16 dimensions

# ...

# Head 8 → 16 dimensions

# Total:

# 8 × 16 = 128

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
  def __init__(self,embed_dim,num_heads):
    super().__init__()
    
    self.embed_dim=embed_dim
    self.num_heads=num_heads
    
    self.head_dim=embed_dim // num_heads
    assert self.head_dim * num_heads==embed_dim
    self.Wq=nn.Linear(embed_dim,embed_dim)
    self.Wk=nn.Linear(embed_dim,embed_dim)
    self.Wv=nn.Linear(embed_dim,embed_dim)
    self.Wo=nn.Linear(embed_dim,embed_dim)
    
    # Step 2 : Start Forward Function
    
  def forward(self,x):
    batch_size , seq_len,_=x.shape # (32,50,128)
    # Step 3 : Generate Q,K,V
    Q=self.Wq(x)
    K = self.Wk(x)
    V = self.Wv(x)
    # Input:
    # (32,50,128)
    # Output:
    # (32,50,128)
    # for all three.
    # Only values change.
    # Shape remains same.
    
    # Step 4 : Split Into Multiple Heads
    
    Q=Q.view(
      batch_size,
      seq_len,
      self.num_heads,
      self.head_dim
    )
    K=K.view(
      batch_size,
      seq_len,
      self.num_heads,
      self.head_dim
    )
    V=V.view(
      batch_size,
      seq_len,
      self.num_heads,
      self.head_dim
    )
    
    # Step 5 : Bring Head Dimension Forward
    # Because attention will now be computed separately for every head.
    Q = Q.transpose(1,2)
    K = K.transpose(1,2)
    V = V.transpose(1,2)
    
      # Step 6 : Compute Attention Scores
      #       Shapes:

      # Q:

      # (32,8,50,16)

      # Kᵀ:

      # (32,8,16,50)

      # Result:

      # (32,8,50,50)
    
    scores = Q @ K.transpose(-2,-1)
    
    # Step 7 : Scale Scores
    scores=scores/math.sqrt(
      self.head_dim
    )
    # Step 8 : Softmax
    attention_weights=F.softmax(
      scores,
      dim=-1
    )
    # Step 9 : Multiply By V
    # Shapes:

    # (32,8,50,50)
    # @
    # (32,8,50,16)
    # Result:
    # (32,8,50,16)
    out = attention_weights @ V
    
    # Step 10 : Restore Original Layout
    # currently (32,8,50,16)
    # we want (32,50,8,16)
    out = out.transpose(1,2)
    
    # Step 11 : Merge Heads
    out = out.contiguous().view(
        batch_size,
        seq_len,
        self.embed_dim
    )
    # Before:

    # (32,50,8,16)
    # After:
    # (32,50,128)
    
    # Step 12 : Final Projection  
    out=self.Wo(out)
    return out
      

      

# Step 1 : Create FFN Class

In [47]:
class FeedForward(nn.Module):
  def __init__(
    self,embed_dim,
    hidden_dim
  ):
    super().__init__()
    self.fc1=nn.Linear(
      embed_dim,
      hidden_dim
    )
    self.fc2=nn.Linear(
      hidden_dim,
      embed_dim
    )
    self.gelu=nn.GELU()
    
  def forward(self,x):
    x=self.fc1(x)
    x=self.gelu(x)
    x=self.fc2(x)
    
    return x

TEST

In [48]:
ffn = FeedForward(
    embed_dim=128,
    hidden_dim=512
)

x = torch.randn(
    32,
    50,
    128
)

out = ffn(x)

print(out.shape)

# At this point we have:

# MultiHeadAttention

# and

# FeedForward

# implemented.

torch.Size([32, 50, 128])


## PART 6 : LayerNorm

## Create LayerNorm

In [49]:
layer_norm=nn.LayerNorm(
  128
)

# Suppose input:

# (32,50,128)

# Output:

# (32,50,128)

# Shape never changes.

# Only values are normalized.

# PART 7 : Residual Connections

This is one of the most important ideas in deep learning.

Instead of:

out = attention(x)

Transformer uses:

out = x + attention(x)

Now we combine:

MultiHeadAttention
+
FeedForward
+
Residual Connections
+
LayerNorm

This exact idea is repeated:

12 times in GPT-2

32 times in LLaMA

80+ times in large models

# Step 1 : Create TransformerBlock Class


What Happens Here?

Suppose:

embed_dim = 128
num_heads = 8
hidden_dim = 512

In [50]:
class TransformerBlock(nn.Module):
  def __init__(
    self,embed_dim,num_heads,hidden_dim
  ):
    super().__init__()
    
    self.attention=MultiHeadAttention(
      embed_dim,
      num_heads
    )
    self.ffn=FeedForward(
      embed_dim,
      hidden_dim
    )
    self.norm1=nn.LayerNorm(
      embed_dim
    )
    self.norm2=nn.LayerNorm(
      embed_dim
    )
  def forward(self,x):
    attention_output=self.attention(x)
    x=x+attention_output # residual connection
    x=self.norm1(x) # normalsiation 1
    ffn_output=self.ffn(x)
    x=x+ffn_output
    x=self.norm2(x)
    
    return x
    

# test transformer block

In [51]:
block = TransformerBlock(
    embed_dim=128,
    num_heads=8,
    hidden_dim=512
)

x = torch.randn(
    32,
    50,
    128
)

# Fix required in MultiHeadAttention.forward (Cell 25):
# add `return out` after `out = self.Wo(out)`

out = block(x)

print(out.shape)

torch.Size([32, 50, 128])


# PART 8 : Transformer Encoder

The idea is simple.

Instead of:

Input
↓
One Block
↓
Output

we do:

Input
↓
Block 1
↓
Block 2
↓
Block 3
↓
Block 4
↓
Output

Each block refines the representation further.

# Step 1 : Create Encoder Class

In [53]:
class TransformerEncoder(nn.Module):
  def __init__(
    self,
    num_layers,
    embed_dim,
    num_heads,
    hidden_dim
  ):
    super().__init__()
    
    self.layers= nn.ModuleList(
      [
        TransformerBlock(
          embed_dim,
          num_heads,
          hidden_dim
        )
        for _ in range(num_layers)
      ]
    )
    
  def forward(self,x):
    for layer in self.layers:
      x=layer(x)
    return x

# test encoder

In [54]:
encoder = TransformerEncoder(
    num_layers=4,
    embed_dim=128,
    num_heads=8,
    hidden_dim=512
)

x = torch.randn(
    32,
    50,
    128
)

out = encoder(x)

print(out.shape)

torch.Size([32, 50, 128])


# What Have We Built?

Essentially:

Embedding
↓
Transformer Block
↓
Transformer Block
↓
Transformer Block
↓
Transformer Block
↓
Output

This is the core idea behind:

BERT
Vision Transformer (ViT)
Encoder side of T5
But GPT Is Different

GPT does NOT use:

Encoder
+
Decoder

from the original Transformer paper.

GPT uses:

Decoder Only

architecture.

So next we will build:

class GPT(nn.Module)